# Data Challenge : Lynred data

---

## Imports

In [13]:
# --- Standard Library ---
import os
import glob

# --- Math & Image Processing ---
import numpy as np
import cv2
from skimage import io
from scipy.ndimage import convolve1d, gaussian_filter1d

# --- Parallelization ---
from joblib import Parallel, delayed

# --- Visualization ---
import matplotlib.pyplot as plt

---

## Load Image

In [14]:
def load_img(path):
    """
    Loads an image from the given file path.

    Args:
        path (str): Full path to the image file.

    Returns:
        np.ndarray: The image as a numpy array.
    """
    # Simply read and return the image
    return io.imread(path)

---

## Build all the data set paths

In [15]:
def load_dataset(folder='train', high_dyn=True):
    """
    Builds a dictionary of image paths grouped by type, sequence, and dynamics.

    Args:
        folder (str): Target directory name.
        high_dyn (bool): Whether to include 'high_dyn' in the search.

    Returns:
        tuple: (Dataset dictionary, Flat list of all image paths)
    """
    cam_types = ['HD', 'SXGA', 'VGA']
    seqs = ['sequence_1', 'sequence_2', 'sequence_3']
    
    # Define dynamics based on the high_dyn flag
    dyns = [
        'low dyn', 
        'low dyn with columns 1', 
        'low dyn with columns 2', 
        'low dyn with columns 3'
    ]
    if high_dyn:
        dyns.insert(0, 'high_dyn')

    data_dict = {}
    all_paths = []

    # Build the dictionary and flat list
    for t in cam_types:
        data_dict[t] = {}
        for seq in seqs:
            data_dict[t][seq] = {}
            for dyn in dyns:
                # Grab all PNG files in the specific folder
                pattern = f"{folder}/{t}/{seq}/{dyn}/*.png"
                paths = glob.glob(pattern)
                
                data_dict[t][seq][dyn] = paths
                all_paths.extend(paths)

    return data_dict, all_paths

--- 

## Correction

In [ ]:
def make_mask(shape, defects):
    """
    Creates a 2D boolean mask for defective pixels based on coordinates.

    Args:
        shape (tuple): The (height, width) of the target image.
        defects (dict): Keys are x-coords, values are dicts with 'start' and 'stop' y-coords.
            
    Returns:
        np.ndarray: 2D boolean array (True = defect).
    """
    h, w = shape
    mask = np.zeros(shape, dtype=bool)
    
    # Return empty mask if no defects are provided
    if not defects:
        return mask

    for x, intervals in defects.items():
        # Skip if x-coordinate is out of image bounds
        if not (0 <= x < w):
            continue
        
        starts = intervals.get('start', [])
        stops = intervals.get('stop', [])
        
        # Apply True to the mask for each vertical interval on this column
        for y0, y1 in zip(starts, stops):
            y_start = max(0, int(y0))
            y_stop = min(h, int(y1))
            mask[y_start:y_stop, x] = True
            
    return mask

In [17]:
def fix_stripes(img, defects):
    """
    Corrects column defects using frequency separation and 1D interpolation.
    Non-defective pixels are preserved completely.

    Args:
        img (np.ndarray): 2D input image.
        defects (dict): Defect coordinates dictionary.
            
    Returns:
        np.ndarray: Corrected image (16-bit uint).
    """
    out_img = np.copy(img)
    h, w = out_img.shape
    
    # 1. Generate the exact 2D mask
    mask = make_mask(out_img.shape, defects)
                    
    if not np.any(mask):
        return img.astype(np.uint16)

    img_float = img.astype(np.float32)

    # 2. Vertical Frequency Separation
    # Isolate low frequencies (smooth background) and high frequencies (details)
    low_freq = gaussian_filter1d(img_float, sigma=11.0, axis=0)
    high_freq = img_float - low_freq 

    low_fixed = np.copy(low_freq)
    
    # 3. Horizontal Interpolation on Low Frequencies
    for y in range(h):
        row_mask = mask[y, :]
        if not np.any(row_mask):
            continue
            
        clean_idx = np.where(~row_mask)[0]
        err_idx = np.where(row_mask)[0]
        
        # Interpolate missing low-frequency pixels using clean neighbors
        if len(clean_idx) > 1:
            interp_vals = np.interp(err_idx, clean_idx, low_freq[y, clean_idx])
            low_fixed[y, err_idx] = interp_vals

    # 4. Recombine and format output
    final_float = low_fixed + high_freq
    fixed_16b = np.clip(np.round(final_float), 0, 65535).astype(np.uint16)
    
    # 5. Apply corrections only to defective areas
    out_img[mask] = fixed_16b[mask]
    
    return out_img.astype(np.uint16)

--- 

## Detection

In [18]:
def detect_defects(img, win_size=50, metric='mean', std_factor=1.0):
    """
    Detects full-column defects based on a 1D local thresholding.

    Args:
        img (np.ndarray): 2D image.
        win_size (int): Size of the 1D convolution window.
        metric (str): 'mean', 'median', or 'std'.
        std_factor (float): Multiplier for the tolerance margin.
        plot (bool): If True, displays a matplotlib graph of the results.

    Returns:
        dict: Defect coordinates structured for fix_stripes().
    """
    h = img.shape[0]

    # 1. Compute the chosen metric across columns
    if metric == 'mean':
        vals = np.mean(img, axis=0)
    elif metric == 'median':
        vals = np.median(img, axis=0)
    elif metric == 'std':
        vals = np.std(img, axis=0)
    else: 
        print('Error: Invalid metric')
        return {}

    # 2. Calculate local trend via moving average
    kernel = np.ones(win_size) / win_size
    trend = convolve1d(vals, kernel, mode='reflect')
    
    # 3. Define threshold margins
    tolerance = std_factor * np.std(vals)
    upper_bound = trend + tolerance
    lower_bound = trend - tolerance
    
    # 4. Bilateral detection
    bad_cols = np.where(np.abs(vals - trend) > tolerance)[0]

    # Format the output dict
    defects = {int(c): {'start': [0], 'stop': [h]} for c in bad_cols}
        
    return defects

---

## Run

In [20]:
# --- Configuration by Image Type ---
PARAMS = {
    'VGA':  {'win_size': 16, 'metric': 'mean', 'std_factor': 1.918},
    'HD':   {'win_size': 24, 'metric': 'mean', 'std_factor': 1.26},
    'SXGA': {'win_size': 31, 'metric': 'mean', 'std_factor': 1.74}
}

def process_img(cam_type, seq, dyn, img_path):
    """
    Processes a single image: loads, detects defects, fixes them, and saves the result.

    Args:
        cam_type (str): 'VGA', 'SXGA', or 'HD'.
        seq (str): Sequence name.
        dyn (str): Dynamics type.
        img_path (str): Full path to the input image.
    """
    filename = os.path.basename(img_path)
    folder = os.path.dirname(img_path)
    
    # Setup results directory
    res_folder = os.path.join(folder, "results")
    os.makedirs(res_folder, exist_ok=True)
    save_path = os.path.join(res_folder, filename)
    
    # 1. Load Image
    img = load_img(img_path)
    
    # 2. Get specific parameters
    p = PARAMS.get(cam_type, {'win_size': 50, 'metric': 'mean', 'std_factor': 1.0, 'n_bands': 1})
    
    # 3. Detect Defects
    defects = detect_defects(
        img, 
        win_size=p['win_size'], 
        metric=p['metric'], 
        std_factor=p['std_factor']
    )
    
    # 4. Correct Image if defects are found
    if not defects:
        fixed_img = np.copy(img)
    else:
        fixed_img = fix_stripes(img, defects)

    # 5. Format and Save
    out_img = np.clip(fixed_img, 0, 65535).astype(np.uint16)
    
    # Double write to prevent OS caching/disk write errors
    cv2.imwrite(save_path, out_img)
    success = cv2.imwrite(save_path, out_img)
    
    if not success:
        print(f"ERROR: OpenCV failed to save {save_path}")
        return None

# ==========================================
# --- MAIN RUN SCRIPT ---
# ==========================================

# 1. Load Dataset
data_dict, _ = load_dataset(folder='train', high_dyn=False)

# 2. Prepare Task List
tasks = []
for t, seqs in data_dict.items():
    for seq, dyns in seqs.items():
        for dyn, paths in dyns.items():
            if dyn == "low dyn":
                continue 
            for path in paths:
                tasks.append((t, seq, dyn, path))

print(f"Launching Joblib for {len(tasks)} images...")

# 3. Parallel Execution
raw_results = Parallel(n_jobs=-1)(
    delayed(process_img)(*task) for task in tasks
)

Launching Joblib for 8607 images...


KeyboardInterrupt: 

---

## Evaluation

In [ ]:
from metrics import evaluate_sequence
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- Configuration ---
root_path = "train" 
sensors   = ["HD", "SXGA", "VGA"]
csv_path  = "results.csv"

# 1. Task Construction
tasks = []
for sensor in sensors:
    sensor_path = os.path.join(root_path, sensor)
    if os.path.exists(sensor_path):
        for entry in os.scandir(sensor_path):
            if entry.is_dir():
                for sim in (1, 2, 3):
                    tasks.append((sensor, entry.path, sim))

print(f"Starting evaluation for {len(tasks)} tasks...\n" + "-"*60)

results = []
final_scores = []

# 2. Parallel Execution with REAL-TIME display
with ProcessPoolExecutor() as pool:
    # Submit all tasks to the pool
    futures = {pool.submit(evaluate_sequence, task): task for task in tasks}
    
    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            results.append(res)
            final_scores.append(float(res[13]))
            print(res[-1])

print("-" * 60)

# 3. CSV Writing once everything is finished
if results:
    with open(csv_path, mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([
            "sensor", "sequence", "def_path", "TP", "FP", "FN",
            "precision", "recall", "F1",
            "RMSE_def", "RMSE_ok", "RMSE_def_norm",
            "RMSE_ok_norm", "final_score"
        ])
        for r in results:
            writer.writerow(r[:-1])

# 4. Final Score Display
if final_scores:
    mean_score = np.mean(final_scores)
    print("\nEvaluation completed! CSV file generated.")
    print("=== MEAN FINAL SCORE ACROSS ALL SEQUENCES ===")
    print(f"Mean score = {mean_score:.6f}")
else:
    print("\nNo results were produced.")

🚀 Lancement de l'évaluation pour 27 tâches...
------------------------------------------------------------
[HD/sequence_1 3] TP=0.007 FP=0.012 FN=0.014 Prec=0.369 Rec=0.337 F1=0.352 RMSE_def=12.55 RMSE_ok=1.81 RMSE_def_norm=0.69 RMSE_ok_norm=0.95 Score final : 0.6612
[HD/sequence_1 2] TP=0.003 FP=0.011 FN=0.003 Prec=0.208 Rec=0.487 F1=0.292 RMSE_def=6.56 RMSE_ok=1.89 RMSE_def_norm=0.84 RMSE_ok_norm=0.95 Score final : 0.6896
[HD/sequence_1 1] TP=0.006 FP=0.011 FN=0.002 Prec=0.371 Rec=0.790 F1=0.505 RMSE_def=4.61 RMSE_ok=1.83 RMSE_def_norm=0.88 RMSE_ok_norm=0.95 Score final : 0.7785
[HD/sequence_2 2] TP=0.004 FP=0.000 FN=0.008 Prec=0.999 Rec=0.319 F1=0.483 RMSE_def=13.29 RMSE_ok=0.01 RMSE_def_norm=0.67 RMSE_ok_norm=1.00 Score final : 0.7146
[HD/sequence_2 3] TP=0.005 FP=0.001 FN=0.008 Prec=0.859 Rec=0.362 F1=0.509 RMSE_def=10.12 RMSE_ok=0.16 RMSE_def_norm=0.75 RMSE_ok_norm=1.00 Score final : 0.7483
[HD/sequence_2 1] TP=0.009 FP=0.002 FN=0.003 Prec=0.821 Rec=0.761 F1=0.790 RMSE_def=6.41 R